Пример скрипта


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, TimestampType, IntegerType, DoubleType, BooleanType
import traceback
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


In [3]:
def create_spark_session(s3_config=None):
    """
    Create and configure a Spark session.

    Parameters
    ----------
    s3_config : dict, optional
        Dictionary containing S3 configuration parameters
        (endpoint_url, access_key, secret_key)

    Returns
    -------
    SparkSession
        Configured Spark session
    """
    print("DEBUG: Начинаем создание Spark сессии")
    try:
        # Создаем базовый Builder
        builder = (SparkSession
            .builder
            .appName("FraudDetectionModel")
        )

        # Если передана конфигурация S3, добавляем настройки
        if s3_config and all(k in s3_config for k in ['endpoint_url', 'access_key', 'secret_key']):
            print(f"DEBUG: Настраиваем S3 с endpoint_url: {s3_config['endpoint_url']}")
            builder = (builder
                .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
                .config("spark.hadoop.fs.s3a.endpoint", s3_config['endpoint_url'])
                .config("spark.hadoop.fs.s3a.access.key", s3_config['access_key'])
                .config("spark.hadoop.fs.s3a.secret.key", s3_config['secret_key'])
                .config("spark.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
            )

        print("DEBUG: Spark сессия успешно сконфигурирована")
        # Создаем и возвращаем сессию Spark
        return builder
    except Exception as e:
        print(f"ERROR: Ошибка создания Spark сессии: {str(e)}")
        print(f"Traceback: {traceback.format_exc()}")

In [4]:
def clean_convert(spark, source_path: str, output_path: str) -> None:

    # Define the schema
    schema = StructType([
        StructField("transaction_id", LongType(), False),
        StructField("tx_datetime", TimestampType(), False),
        StructField("customer_id", IntegerType(), False),
        StructField("terminal_id", IntegerType(), False),
        StructField("tx_amount", DoubleType(), False),
        StructField("tx_time_seconds", LongType(), False),
        StructField("tx_time_days", IntegerType(), False),
        StructField("tx_fraud", IntegerType(), False),
        StructField("tx_fraud_scenario", IntegerType(), False)
    ])


    print("Try to read CSV files from source bucket...")

    #Read the TXT file as a CSV file
    df_txt = spark.read.csv(
        source_path,
        header=False,
        comment="#",  # comment character
        schema=schema,
        sep=",",  # separator (comma in this case)
        mode="PERMISSIVE"  # Handles lines with more or fewer columns.
    )
    df = df_txt


    # Clean the DataFrame by:
    # 1. Dropping rows where all columns have null values.
    # 2. Removing duplicate rows.
    # 3. Filtering rows to include only those with a positive `tx_amount`.
    df_cleaned = df.na.drop(how="all").distinct().filter(df.tx_amount > 0)

    # Save the cleaned DataFrame as a Parquet file
    df_cleaned.repartition(10).write.mode("overwrite").parquet(output_path)

    # print('Records count after clean:', df_cleaned.count())
    #
    # # Stop the Spark session
    # spark.stop()
    print("Successfully saved the result to the output bucket!")


In [8]:
def read_parquet_and_split(spark, parquet_path: str, train_ratio: float = 0.8):
    """
    Reads a Parquet file, splits it into train and test DataFrames.

    Args:
        spark: The SparkSession.
        parquet_path: The path to the Parquet file.
        train_ratio: The ratio of data to use for training (default: 0.8).

    Returns:
        A tuple containing the train and test DataFrames.
    """
    try:
        df = spark.read.parquet(parquet_path)

        # Split the DataFrame into train and test sets
        train_df, test_df = df.randomSplit([train_ratio, 1 - train_ratio], seed=42)
        print(f"Training set size: {train_df.count()}")
        print(f"Testing set size: {test_df.count()}")
        return train_df, test_df

    except Exception as e:
        print(f"Error reading and splitting Parquet file: {e}")
        traceback.print_exc()
        return None, None

In [5]:
spark_session = create_spark_session().getOrCreate()

DEBUG: Начинаем создание Spark сессии
DEBUG: Spark сессия успешно сконфигурирована


In [6]:
file_path = "data/2022-09-05.txt"
output_path = "data/2022-09-05.parquet"
clean_convert(spark_session, file_path, output_path)

Try to read CSV files from source bucket...
Successfully saved the result to the output bucket!


In [9]:
# Read the Parquet file and split into train and test sets
train_data, test_data = read_parquet_and_split(spark_session, output_path)

Training set size: 37594758
Testing set size: 9398212


In [ ]:
#spark_session.stop() #Only stop when you are done with the spark session.

In [6]:


def load_data(spark, input_path):
    """
    Load and prepare the fraud detection dataset.

    Parameters
    ----------
    spark : SparkSession
        Spark session
    input_path : str
        Path to the input data

    Returns
    -------
    tuple
        (train_df, test_df) - Spark DataFrames for training and testing
    """

    schema = StructType([
      StructField("transaction_id", LongType(), True),
      StructField("tx_datetime", TimestampType(), True),
      StructField("customer_id", IntegerType(), True),
      StructField("terminal_id", IntegerType(), True),
      StructField("tx_amount", DoubleType(), True),
      StructField("tx_time_seconds", LongType(), True),
      StructField("tx_time_days", IntegerType(), True),
      StructField("tx_fraud", IntegerType(), True),
      StructField("tx_fraud_scenario", IntegerType(), True)
    ])
    print(f"DEBUG: Начинаем загрузку данных из: {input_path}")
    try:
        # Load the data
        print(f"DEBUG: Чтение CSV файла из {input_path}")
       # df = spark.read.csv(input_path, header=True, inferSchema=True)
        df = spark.read.csv(
            input_path,
            header=False,
            comment="#",  # comment character
            schema=schema,
            sep=",",       # separator (comma in this case)
            mode="PERMISSIVE" # Handles lines with more or fewer columns.
        )

        # Print schema and basic statistics
        print("Dataset Schema:")
        df.printSchema()
        print(f"Total records: {df.count()}")

        # Проверим первые несколько строк
        print("DEBUG: Первые 5 строк данных:")
        df.show(5, truncate=False)

        # Split the data into training and testing sets
        print("DEBUG: Разделение на обучающую и тестовую выборки")
        train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
        print(f"Training set size: {train_df.count()}")
        print(f"Testing set size: {test_df.count()}")

        return train_df, test_df
    except Exception as e:
        print(f"ERROR: Ошибка загрузки данных: {str(e)}")
        print(f"Traceback: {traceback.format_exc()}")
        raise

In [7]:
def prepare_features(train_df, test_df):
    """
    Prepare features for model training.

    Parameters
    ----------
    train_df : DataFrame
        Training DataFrame
    test_df : DataFrame
        Testing DataFrame

    Returns
    -------
    tuple
        (train_df, test_df, feature_cols) - Prepared DataFrames and feature column names
    """
    print("DEBUG: Начинаем подготовку признаков")
    try:
        # Получаем типы столбцов
        print("DEBUG: Проверяем типы столбцов")
        dtypes = dict(train_df.dtypes)
        print(f"DEBUG: Типы данных: {dtypes}")

        # Исключаем строковые столбцы и целевую переменную 'fraud'
        # feature_cols = [col for col in train_df.columns
        #                 if col != 'tx_fraud' and col != 'tx_fraud_scenario' ]
        feature_cols = ['customer_id', 'terminal_id', 'tx_amount']
        print(f"DEBUG: Выбрано {len(feature_cols)} числовых признаков: {feature_cols}")

        # Проверим наличие нулевых значений
        print("DEBUG: Проверка наличия нулевых значений в обучающей выборке")
        for col in train_df.columns:
            null_count = train_df.filter(train_df[col].isNull()).count()
            if null_count > 0:
                print(f"WARNING: Колонка '{col}' содержит {null_count} нулевых значений")

        return train_df, test_df, feature_cols
    except Exception as e:
        print(f"ERROR: Ошибка подготовки признаков: {str(e)}")
        print(f"Traceback: {traceback.format_exc()}")
        raise

In [13]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

def train_model(train_df, test_df, feature_cols, model_type="rf"):
    """
    Train a fraud detection model (simplified version without MLflow and hyperparameter tuning)

    Parameters
    ----------
    train_df : DataFrame
        Training DataFrame
    test_df : DataFrame
        Testing DataFrame
    feature_cols : list
        List of feature column names
    model_type : str
        Model type to train ('rf' for Random Forest, 'lr' for Logistic Regression)

    Returns
    -------
    tuple
        (trained_model, metrics) - Trained model and its performance metrics
    """
    print(f"DEBUG: Начинаем обучение модели типа {model_type}")

    try:
        # Create feature vector
        print("DEBUG: Создание преобразователя признаков")
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
        scaler = StandardScaler(
            inputCol="features_raw",
            outputCol="features",
            withStd=True,
            withMean=True
        )

        # Select model based on type
        print("DEBUG: Создание классификатора")
        if model_type == "rf":
            classifier = RandomForestClassifier(
                labelCol="fraud",
                featuresCol="features",
                numTrees=10,  # Фиксированное значение вместо перебора
                maxDepth=5    # Фиксированное значение вместо перебора
            )
        else:
            raise ValueError(f"Unsupported model type: {model_type}")

        # Create pipeline
        print("DEBUG: Создание пайплайна")
        pipeline = Pipeline(stages=[assembler, scaler, classifier])

        # Create evaluators
        print("DEBUG: Создание оценщиков")
        evaluator_auc = BinaryClassificationEvaluator(
            labelCol="fraud",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        )
        evaluator_acc = MulticlassClassificationEvaluator(
            labelCol="fraud",
            predictionCol="prediction",
            metricName="accuracy"
        )
        evaluator_f1 = MulticlassClassificationEvaluator(
            labelCol="fraud",
            predictionCol="prediction",
            metricName="f1"
        )

        # Train the model
        print("DEBUG: Обучаем модель...")
        trained_model = pipeline.fit(train_df)
        print("DEBUG: Модель успешно обучена")

        # Make predictions on test data
        print("DEBUG: Делаем предсказания на тестовых данных")
        predictions = trained_model.transform(test_df)
        print("DEBUG: Предсказания получены")

        # Calculate metrics
        print("DEBUG: Рассчитываем метрики")
        auc = evaluator_auc.evaluate(predictions)
        accuracy = evaluator_acc.evaluate(predictions)
        f1 = evaluator_f1.evaluate(predictions)

        # Print metrics
        print(f"AUC: {auc}")
        print(f"Accuracy: {accuracy}")
        print(f"F1 Score: {f1}")

        metrics = {
            "auc": auc,
            "accuracy": accuracy,
            "f1": f1
        }

        return trained_model, metrics

    except Exception as e:
        print(f"ERROR: Ошибка обучения модели: {str(e)}")
        raise


In [8]:
spark_session = create_spark_session().getOrCreate()


DEBUG: Начинаем создание Spark сессии
DEBUG: Spark сессия успешно сконфигурирована


In [9]:
file_path = "data/2022-09-05.txt"
train_df, test_df = load_data(spark_session,file_path )

DEBUG: Начинаем загрузку данных из: data/2022-09-05.txt
DEBUG: Чтение CSV файла из data/2022-09-05.txt
Dataset Schema:
root
 |-- transaction_id: long (nullable = true)
 |-- tx_datetime: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: integer (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: long (nullable = true)
 |-- tx_time_days: integer (nullable = true)
 |-- tx_fraud: integer (nullable = true)
 |-- tx_fraud_scenario: integer (nullable = true)

Total records: 46993904
DEBUG: Первые 5 строк данных:
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|1738801610    |2022-09-05 14:50:42|0   

In [10]:
train_df, test_df, feature_cols = prepare_features(train_df, test_df)

DEBUG: Начинаем подготовку признаков
DEBUG: Проверяем типы столбцов
DEBUG: Типы данных: {'transaction_id': 'bigint', 'tx_datetime': 'timestamp', 'customer_id': 'int', 'terminal_id': 'int', 'tx_amount': 'double', 'tx_time_seconds': 'bigint', 'tx_time_days': 'int', 'tx_fraud': 'int', 'tx_fraud_scenario': 'int'}
DEBUG: Выбрано 3 числовых признаков: ['customer_id', 'terminal_id', 'tx_amount']
DEBUG: Проверка наличия нулевых значений в обучающей выборке


In [14]:
model, metrics = train_model(train_df, test_df, feature_cols)

DEBUG: Начинаем обучение модели типа rf
DEBUG: Создание преобразователя признаков
DEBUG: Создание классификатора
DEBUG: Создание пайплайна
DEBUG: Создание оценщиков
DEBUG: Обучаем модель...
ERROR: Ошибка обучения модели: An error occurred while calling o123.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 25.0 failed 4 times, most recent failure: Lost task 1.3 in stage 25.0 (TID 322, rc1a-dataproc-d-uapc2dosc2s6r4b3.mdb.yandexcloud.net, executor 4): org.apache.spark.SparkException: Failed to execute user defined function(VectorAssembler$$Lambda$2968/864306755: (struct<customer_id_double_VectorAssembler_25faac8da89d:double,terminal_id_double_VectorAssembler_25faac8da89d:double,tx_amount:double>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.has

Py4JJavaError: An error occurred while calling o123.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 25.0 failed 4 times, most recent failure: Lost task 1.3 in stage 25.0 (TID 322, rc1a-dataproc-d-uapc2dosc2s6r4b3.mdb.yandexcloud.net, executor 4): org.apache.spark.SparkException: Failed to execute user defined function(VectorAssembler$$Lambda$2968/864306755: (struct<customer_id_double_VectorAssembler_25faac8da89d:double,terminal_id_double_VectorAssembler_25faac8da89d:double,tx_amount:double>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenExec$$anon$1.hasNext(WholeStageCodegenExec.scala:729)
	at org.apache.spark.sql.execution.aggregate.ObjectAggregationIter